# CLM-0.3 — Progressive Growth

Preflight and formal experiment notebook. Formal execution remains opt-in.

In [ ]:
from pathlib import Path
import json, sys, hashlib, subprocess

ROOT = Path('/kaggle/working/mini-cells')
REPO_URL = 'https://github.com/ArcheLabs/mini-cells.git'
BRANCH = 'codex/clm-0.3-progressive-growth'

# Kaggle kernels do not automatically materialize the GitHub repository.
# Bootstrap it explicitly on a fresh session; reuse an existing checkout on resume.
if not (ROOT / '.git').exists():
    if ROOT.exists() and any(ROOT.iterdir()):
        raise RuntimeError(f'{ROOT} exists but is not a git checkout; remove or rename it before continuing')
    ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(ROOT)
    ], check=True)
else:
    print('Reusing existing Kaggle checkout:', ROOT)

CODE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True).strip()
print('repository:', ROOT)
print('branch:', BRANCH)
print('commit:', CODE_COMMIT)

sys.path.insert(0, str(ROOT / 'research'))
print('research path:', sys.path[0])

In [ ]:
import torch
print({'python': sys.version.split()[0], 'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]})
release = ROOT / 'artifacts/releases/clm-0.1/model.pt'
if not release.is_file():
    raise FileNotFoundError(f'CLM-0.1 release artifact missing after repository bootstrap: {release}')
observed = hashlib.sha256(release.read_bytes()).hexdigest()
assert observed == '87d36c408ae3873ffd567ebf17050661b42ddae2c8d5d1bab84b2c27c3c7e7a0'
print('CLM-0.1 SHA-256:', observed)

In [ ]:
# Mandatory preflight. Do not start the formal matrix unless this passes.
subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_clm_progressive_growth.py', 'tests/test_growth_router.py', 'tests/test_growth_checkpoint.py', '-q'], cwd=ROOT, check=True)

In [ ]:
from minicells.clm_growth import ProgressiveGrowthCLM
model = ProgressiveGrowthCLM.from_clm01_release(str(ROOT / 'artifacts/releases/clm-0.1'))
print('zero-birth expert counts:', model.expert_counts_by_stage())

## Formal matrix

The safety switch below is false by default. The parent runner launches the paired 3×3 matrix, monitors GPUs, and aggregates matched fixed-4 controls only after all workers complete.

In [ ]:
RUN_FORMAL = False
RESULTS = ROOT / 'results/clm-0.3-progressive-growth'
runner = [sys.executable, 'scripts/run_clm_progressive_growth_001.py', '--output-root', str(RESULTS)]
if RUN_FORMAL:
    subprocess.run([*runner, '--execute'], cwd=ROOT, check=True)
else:
    subprocess.run(runner, cwd=ROOT, check=True)
    print('NO DATA / PREFLIGHT ONLY')

In [ ]:
# The parent runner writes these only after matched aggregation succeeds.
decision_path = RESULTS / 'decision.json'
history_path = RESULTS / 'formal-ppl-history.csv'
summary_path = RESULTS / 'replicate-summary.json'
if decision_path.exists():
    decision = json.loads(decision_path.read_text())
    print(json.dumps(decision, indent=2, sort_keys=True))
    print('formal history:', history_path)
    print('replicate summary:', summary_path)
else:
    print('NO DATA / PREFLIGHT ONLY; matched ratios and formal decisions unavailable')